# PubMed

There are articles without a title, an abstract, or both across all datasets.

To maximize the number of articles with text data for classification, we query the PubMed database.
Similarly to OpenAlex, we query PubMed through its eutils API for each article with a PubMed identifier.

We further incorporate additional metadata, which the API provides, into the datasets.

> **NOTE**
>
> The datasets after running this notebook already exist within the ``data/datasets/03_pubmed`` directory.
> If you simply want to inspect the data, you can directly access them there.
>
> When running the code for comprehensiveness, keep in mind that the downloads might take a while, depending on your bandwidth.

## Custom Modules
The ``src`` directory houses custom modules with functions that will be reused throughout the project.

To be able to import these moduls, we begin with programmatically adding the project's root directory to ``sys.path``.

After adding the root to ``sys.path``, we can import the ``data`` and ``util`` modules:

In [1]:
import os, sys

# recursively search for the root directory containing a specific file
def find_root_dir(search_for='.gitignore'):

    current_dir = os.getcwd()

    while True:
        if os.path.exists(os.path.join(current_dir, search_for)):
            return current_dir
        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find '{search_for}' in any parent directory.")
        current_dir = parent_dir


# save the root directory to a variable
root_dir = find_root_dir()
print(f"Root directory found: {root_dir}")

# add the root directory to the system path
sys.path.append(root_dir)
if root_dir in sys.path:
    print(f"Root directory added to system path.")


# import custom modules
from src import data

Root directory found: c:\dev\automated_title_abstract_screening
Root directory added to system path.


## PubMed API

Without an API key, the PubMedFetcher interface is limited to three queries per second.
To speed up the retrieval of article metadata we recommend setting up an API key.

You can grab an API key from [here](https://ncbiinsights.ncbi.nlm.nih.gov/2017/11/02/new-api-keys-for-the-e-utilities/)

Now simply create a ``.env`` file in the root directory of your project and store the key in the format ``NCBI=yourapikeygoeshere``

When you have an API key set, you can execute the next cell to pass the key to the environment, from where PubMedFetcher can access it:

In [2]:
from dotenv import load_dotenv # read .env files

# import the ncbi api key from a local .env file
load_dotenv()
ncbi_api_key = os.getenv('NCBI')

#set the api key as an environment variable to increase the rate limit
if ncbi_api_key is not None:
    os.environ['NCBI_API_KEY'] = ncbi_api_key
    if os.environ['NCBI_API_KEY']:
        print("Added the NCBI API key to the environment variables.")

Added the NCBI API key to the environment variables.


# Load Datasets
Next, we load the datasets with texts from OpenAlex to a dictionary as before:

In [3]:
import polars as pl
from src import data

# the directory with the datasets after the call to OpenAlex
data_directory_openalex = '../../../data/datasets/02_openalex'

# create the dictionary of datasets
datasets = data.dict_from_directory(
    directory=data_directory_openalex,
    type='polars',
)

# type checking - dataframes should be polars dataframes
assert isinstance(datasets, dict), 'datasets should be a dictionary of polars dataframes'

# Rename Literature IDs
The SYNERGY datasets refer to ids for PubMed as 'literature_id'. 

Rename to 'pubmed_id' for consistency:

In [4]:
for subject, dataset in datasets.items(): 
    datasets[subject] = dataset.rename({'literature_id': 'pubmed_id'})

Confirm that ``literature_id`` has been renamed to ``pubmed_id``:

In [5]:
datasets['animal_depression'].head(1)

include,title,abstract,doi,pubmed_id,openalex_id
bool,str,str,str,f64,str
false,"""Inhibition of cellular transpo…","""5-Thio-d-glucopyranose, the ne…","""https://doi.org/10.1042/bj1300…",4.656804e6,"""https://openalex.org/W24010252…"


# Type Casting
The PubMed ID is wrongfully formatted as Float64 in the animal depression dataset. 
Cast it to Integer:

In [6]:
if isinstance(datasets['animal_depression'], pl.DataFrame):
    # Convert the 'pubmed_id' column to integer if it is not already
    datasets['animal_depression'] = datasets['animal_depression'].with_columns(
        pl.col('pubmed_id').cast(pl.Int64)
    )

Confirm that pubmed_id is now indeed an integer:

In [7]:
datasets['animal_depression'].head(1)

include,title,abstract,doi,pubmed_id,openalex_id
bool,str,str,str,i64,str
false,"""Inhibition of cellular transpo…","""5-Thio-d-glucopyranose, the ne…","""https://doi.org/10.1042/bj1300…",4656804,"""https://openalex.org/W24010252…"


# Uniform Schema

## Definition
Define a uniform schema with columns and associated data types that all datasets will use from now on:

In [8]:
schema = pl.Schema({
    "include": pl.Boolean,
    "title": pl.String,
    "abstract": pl.String,
    "first_author": pl.String,
    "year": pl.Int16,
    "journal": pl.String,
    "doi": pl.String,
    "pubmed_id": pl.Int64,
    "authors": pl.String,
    "pubmed_type": pl.String,
    "publication_types": pl.String,
    "mesh": pl.String,
})

## Assignment
Assign the schema to all datasets:

In [9]:
from typing import cast 

for subject, dataset in datasets.items():
    
    # Ensure the dataset is a Polars DataFrame
    dataset = cast(pl.DataFrame, dataset)
    
    # Assign the schema to the dataset
    datasets[subject] = pl.concat(items=[pl.DataFrame(schema=schema), dataset], how='diagonal')

# Download Function



We will use the ``metapub`` package, to retrieve article metadata from the eutils API.

When importing the package, you will likely receive a deprecation warning.
This warning stems from the ``eutils`` package, which itself is a dependency of ``metapub``.

The deprecation has already been fixed, but the latest version is yet to be released to PyPi.
For more information see [here](https://github.com/biocommons/eutils/issues/183)

In [10]:
from metapub import PubMedFetcher, PubMedArticle

c:\Users\mfaig\miniconda3\envs\test\Lib\site-packages\eutils\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Define a function which retrieves missing data from PubMed in the following way:

- Download article data for each article that contains a PubMed-ID
- Fill each empty field in the schema with the downloaded data 

In [11]:
from tqdm.notebook import tqdm

def fill_missing_from_pubmed(subject: str, dataframe: pl.DataFrame, schema: pl.Schema) -> pl.DataFrame:
    """
    Fill missing metadata from PubMed for a given subject.
    
    Args:
        subject (str): The subject of the dataset - one of the dictionary keys.
        dataframe (pl.DataFrame): The dataframe containing the data.
        schema (pl.Schema): The schema to use for the dataframe.
        
    Returns:
        pl.DataFrame: The updated dataframe with filled metadata.
    """

    # We will save all previous and added metadata in this dataframe
    frame_with_metadata = pl.DataFrame(schema=schema)

    # Initialize a PubMedFetcher instance
    fetcher = PubMedFetcher()

    # Function to add a single row to the frame_with_metadata dataframe
    def add_row_to_frame(row_data: dict) -> None:
        """
        Add a row to the frame_with_metadata dataframe.
        
        Args:
            row_data (dict): The data to add as a row.
        """

        # Create a new dataframe with one row that matches the schema
        new_row = pl.DataFrame([row_data], schema=schema)
        # Concatenate the new row to the existing dataframe
        nonlocal frame_with_metadata
        frame_with_metadata.extend(new_row)    

    # Define the mapping from dataframe columns to PubMed attributes
    eutils_mapping = {
        "include": "include",
        "title": "title",
        "abstract": "abstract",
        "first_author": "author1_last_fm",
        "year": "year",
        "journal": "journal",
        "doi": "doi",
        "pubmed_id": "pmid",
        "authors": "authors_str",
        "pubmed_type": "pubmed_type",
        "publication_types": "publication_types",
        "mesh": "mesh"
    }

    subject_mapping = {
        'adhd': 'Attention Deficit Hyperactivity Disorder',
        'animal_depression': 'Animal Depression',
        'atypical_antipsychotics': 'Atypical Antipsychotics',
        'calcium_channel_blockers': 'Calcium Channel Blockers',
        'oral_hypoglycemics': 'Oral Hypoglycemics',

    }

    # Search metadata for each row individually
    for row in tqdm(
        iterable=dataframe.iter_rows(named=True),
        desc=f"Retrieving {subject_mapping[subject]} metadata",
        total=dataframe.height,
        leave=True
    ):
        
        # Transform the row to match the schema, fill missing values with None
        row_data = {col: row.get(col, None) for col in schema}

        # Extract the PubMed ID from the row
        pubmed_id = row_data.get('pubmed_id')

        # If there is no PubMed ID, keep the row as is
        if pubmed_id is None:
            add_row_to_frame(row_data)
            continue
        # If there is a PubMed ID, fetch metadata and fill empty fields
        else:
            try:
                # Create a list of columns with empty values in the row
                empty_keys = [k for k, v in row_data.items() if v is None]

                # Fetch the metadata for the current article from eutils
                metadata = fetcher.article_by_pmid(pubmed_id)

                # Fill metadata for each empty key if possible
                for empty_key in empty_keys:
                    
                    # The attributes are named differently in eutils
                    eutils_key = eutils_mapping.get(empty_key)
                    
                    # Check whether the key exists in the metadata
                    if eutils_key and hasattr(metadata, eutils_key):

                        # Get the value from the metadata using the eutils key
                        eutils_value = getattr(metadata, eutils_key, None)

                        # concatenate publication types and mesh into strings
                        if eutils_key == 'publication_types':
                            if isinstance(eutils_value, dict):
                                publication_types = '; '.join([f"{k}: {v}" for k, v in eutils_value.items()])
                                row_data[empty_key] = publication_types if publication_types else None
                            else: continue
                        elif eutils_key == 'mesh':
                            if isinstance(eutils_value, dict):
                                mesh_terms = '; '.join([f"{k}: {v['descriptor_name']}" for k, v in eutils_value.items()])
                                row_data[empty_key] = mesh_terms if mesh_terms else None
                            else: continue
                        else:
                            # Set the value from the metadata if it exists
                            row_data[empty_key] = eutils_value
                    else:
                        # If the key does not exist in the metadata, keep the row as is
                        row_data[empty_key] = None
                
                # Add the row with filled metadata to the frame
                add_row_to_frame(row_data)
            
            # If fetching fails, keep the row as is
            except Exception as e:
                print(f"Error fetching data for PubMed ID {pubmed_id}: {e}")
                add_row_to_frame(row_data)
                continue

    return frame_with_metadata

# Download and Export
Apply the aforementioned function to retrieve missing data.

Save the extended dataframes to .csv files:

In [ ]:
data_directory_uniform =  '../../../data/datasets/03_pubmed'

for subject, dataset in datasets.items():
    
    # Ensure the dataset is a Polars DataFrame
    dataset = cast(pl.DataFrame, dataset)
    
    # download metadata from pubmed eutils
    dataset_with_metadata = fill_missing_from_pubmed(
        subject=subject,
        dataframe=dataset,
        schema=schema
    )
    
    # save the dataframe with metadata to a .csv file
    dataset_with_metadata.write_csv(
        file=f'{data_directory_uniform}/{subject}_pubmed.csv'
    )